In [1]:
import numpy as np

In [6]:
class SkipgramModel:
    def __init__(self, vocab_size, embed_size, learning_rate = 0.1):
        self.v_size = vocab_size
        self.e_size = embed_size
        self.lr = learning_rate

        self.W1 = np.random.randn(self.v_size, self.e_size) * 0.1
        self.W2 = np.random.randn(self.e_size, self.v_size) * 0.1

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
        
    def train_step(self, center_word_idx, pos_idx, neg_size = 5):
        h = self.W1[center_word_idx]
        v_pos = self.W2[:, pos_idx]

        neg_indices = np.random.choice(self.v_size, neg_size)
        v_neg = self.W2[:, neg_indices]

        pred_pos = self._sigmoid(np.dot(h, v_pos))
        pred_neg = self._sigmoid(np.dot(h, v_neg))

        error_pos = pred_pos - 1.0  
        error_neg = pred_neg - 0.0

        grad_v_pos = h * error_pos
        grad_v_neg = np.outer(h, error_neg)

        grad_h = (v_pos * error_pos) + np.dot(v_neg, error_neg)

        self.W2[:, pos_idx] -= self.lr * grad_v_pos
        self.W2[:, neg_indices] -= self.lr * grad_v_neg
        self.W1[center_word_idx] -= self.lr * grad_h

        loss = -(np.log(pred_pos + 1e-9) + np.sum(np.log(1 - pred_neg + 1e-9)))

        return loss